In [18]:
import torch
from torch.utils.data import DataLoader
from transformers import BertConfig, BertForSequenceClassification, BertTokenizer
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm

# 檢查並設定運算設備 (如果有 GPU/MPS 就使用)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"目前使用的運算設備: {device}")

目前使用的運算設備: cuda


In [19]:
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_LENGTH = 256

# 訓練資料筆數 (這裡調小一點供快速測試跑通，實戰時請拿掉這層限制)
TRAIN_SAMPLES = 25000
VAL_SAMPLES = 5000
NUM_LABELS = 2  # 情感分析 (正面/負面)

In [4]:
print("正在載入資料集...")
dataset = load_dataset("imdb")

# 取部分資料作為示範
train_dataset = dataset['train'].shuffle(seed=42).select(range(TRAIN_SAMPLES))
val_dataset = dataset['test'].shuffle(seed=42).select(range(VAL_SAMPLES))

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenization 處理函數
def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print("正在進行文字 Tokenize ...")
train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

# 僅保留模型需要的 Tensor 欄位
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 建立 DataLoader
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print("資料處理完成！")

正在載入資料集...


NameError: name 'load_dataset' is not defined

In [21]:
config = BertConfig(
    vocab_size=30522,            
    hidden_size=256,             
    num_hidden_layers=4,         
    num_attention_heads=8,       
    intermediate_size=1024,      
    max_position_embeddings=512, 
    num_labels=NUM_LABELS
)

# 建立模型
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', 
    num_labels=NUM_LABELS
)
model.to(device)

# 定義優化器
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

print(f"模型已載入至 {device} 並準備就緒。")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


模型已載入至 cuda 並準備就緒。


In [22]:
for epoch in range(EPOCHS):
    print(f"\n======== Epoch {epoch+1} / {EPOCHS} ========")
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc="Training")
    for batch in progress_bar:
        # 1. 將資料移動到對應設備
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # 2. 梯度歸零
        model.zero_grad()
        
        # 3. 前向傳播 (Forward pass)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        
        # 4. 反向傳播 (Backward pass)
        loss.backward()
        
        # 5. 更新模型權重
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # 梯度裁剪防止爆炸
        optimizer.step()
        
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"平均訓練 Loss: {avg_train_loss:.4f}")


======== Epoch 1 / 3 ========


Training:   0%|          | 0/1563 [00:00<?, ?it/s]

平均訓練 Loss: 0.2768

======== Epoch 2 / 3 ========


Training:   0%|          | 0/1563 [00:00<?, ?it/s]

平均訓練 Loss: 0.1659

======== Epoch 3 / 3 ========


Training:   0%|          | 0/1563 [00:00<?, ?it/s]

平均訓練 Loss: 0.0944


In [23]:
print("開始進行驗證...")
model.eval()
val_loss = 0
correct_predictions = 0
total_predictions = 0

with torch.no_grad(): # 推論階段不需要計算梯度以節省記憶體
    for batch in tqdm(val_dataloader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        val_loss += loss.item()
        
        # 計算準確率
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        correct_predictions += (predictions == labels).sum().item()
        total_predictions += labels.size(0)

avg_val_loss = val_loss / len(val_dataloader)
accuracy = correct_predictions / total_predictions

print(f"\n💡 最終驗證結果：")
print(f"驗證 Loss: {avg_val_loss:.4f}")
print(f"準確率 (Accuracy): {accuracy * 100:.2f}%")

開始進行驗證...


Validating:   0%|          | 0/313 [00:00<?, ?it/s]


💡 最終驗證結果：
驗證 Loss: 0.3961
準確率 (Accuracy): 91.28%
